In [1]:
import json
import os
import re
import subprocess
from pathlib import Path
from urllib.parse import urlparse

try:
    import pandas as pd
    PANDAS_AVAILABLE = True
except ImportError:
    PANDAS_AVAILABLE = False

In [2]:
class IntegrityProvenanceChecker:
    """
    IntegrityProvenanceChecker

    This checker evaluates whether a research software artifact supports
    integrity and provenance through traceable, verifiable, and reproducible
    metadata and environment information.

    It checks for:
    - Git/version-control metadata
    - LICENSE
    - CITATION.cff / citation metadata
    - README
    - dependency/environment files
    - lock files
    - Dockerfile/container files
    - CI/build workflow files
    - release/changelog/version files
    - checksums/hashes
    - suspicious untrusted dependency sources

    Formal idea:
        integrityProvenance : A → {True, False}
    """

    def __init__(
        self,
        json_file,
        download_dir="downloads",
        minimum_score=5,
        max_untrusted_sources=0
    ):
        self.json_file = json_file
        self.download_dir = Path(download_dir)
        self.download_dir.mkdir(parents=True, exist_ok=True)

        self.minimum_score = minimum_score
        self.max_untrusted_sources = max_untrusted_sources

        self.artifacts = self.load_metadata(json_file)
        self.results = []

    def load_metadata(self, json_file):
        with open(json_file, "r", encoding="utf-8") as file:
            data = json.load(file)

        return data.get("artifacts", {})

    def is_git_repository(self, uri):
        return isinstance(uri, str) and uri.startswith("https://github.com/")

    def repo_name_from_uri(self, uri):
        parsed = urlparse(uri)
        repo_name = parsed.path.rstrip("/").split("/")[-1]

        if repo_name.endswith(".git"):
            repo_name = repo_name[:-4]

        return repo_name or "repository"

    def clone_repository(self, artifact_id, uri):
        repo_name = self.repo_name_from_uri(uri)
        target_dir = self.download_dir / repo_name

        if target_dir.exists():
            print(f"📁 Repository already exists: {target_dir}")
            return target_dir

        print(f"⬇️ Cloning repository: {uri}")

        try:
            result = subprocess.run(
                ["git", "clone", "--depth", "1", uri, str(target_dir)],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                timeout=120
            )

            if result.returncode != 0:
                print(f"❌ Failed to clone repository for {artifact_id}")
                print(result.stderr.strip())
                return None

            print(f"✅ Cloned to: {target_dir}")
            return target_dir

        except Exception as e:
            print(f"❌ Clone error for {artifact_id}: {e}")
            return None

    def read_text_file(self, path, max_chars=300000):
        try:
            with open(path, "r", encoding="utf-8", errors="ignore") as file:
                return file.read(max_chars)
        except Exception:
            return ""

    def find_existing_files(self, repo_dir, possible_paths):
        found = []

        for relative_path in possible_paths:
            path = repo_dir / relative_path
            if path.exists():
                found.append(relative_path)

        return found

    def find_files_by_name(self, repo_dir, names):
        found = []
        names_lower = {name.lower() for name in names}

        for root, dirs, files in os.walk(repo_dir):
            dirs[:] = [
                d for d in dirs
                if d not in {
                    ".git",
                    "__pycache__",
                    ".pytest_cache",
                    ".mypy_cache",
                    "node_modules",
                    ".venv",
                    "venv",
                    "build",
                    "dist"
                }
            ]

            for file in files:
                if file.lower() in names_lower:
                    path = Path(root) / file
                    try:
                        found.append(str(path.relative_to(repo_dir)))
                    except Exception:
                        found.append(str(path))

        return found

    def find_files_by_extensions(self, repo_dir, extensions):
        found = []

        for root, dirs, files in os.walk(repo_dir):
            dirs[:] = [
                d for d in dirs
                if d not in {
                    ".git",
                    "__pycache__",
                    ".pytest_cache",
                    ".mypy_cache",
                    "node_modules",
                    ".venv",
                    "venv",
                    "build",
                    "dist"
                }
            ]

            for file in files:
                path = Path(root) / file

                if path.suffix.lower() in extensions:
                    found.append(path)

        return found

    def has_git_metadata(self, repo_dir):
        return (repo_dir / ".git").exists()

    def get_git_commit_hash(self, repo_dir):
        try:
            result = subprocess.run(
                ["git", "rev-parse", "HEAD"],
                cwd=str(repo_dir),
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                timeout=20
            )

            if result.returncode == 0:
                return result.stdout.strip()

        except Exception:
            pass

        return None

    def get_git_remote_url(self, repo_dir):
        try:
            result = subprocess.run(
                ["git", "remote", "get-url", "origin"],
                cwd=str(repo_dir),
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                timeout=20
            )

            if result.returncode == 0:
                return result.stdout.strip()

        except Exception:
            pass

        return None

    def detect_metadata_files(self, repo_dir):
        return {
            "readme_files": self.find_existing_files(
                repo_dir,
                ["README.md", "README.rst", "README.txt", "readme.md", "readme.rst", "readme.txt"]
            ),
            "license_files": self.find_existing_files(
                repo_dir,
                ["LICENSE", "LICENSE.md", "LICENSE.txt", "COPYING", "COPYING.txt"]
            ),
            "citation_files": self.find_existing_files(
                repo_dir,
                ["CITATION.cff", "citation.cff", "codemeta.json", "CODEMETA.json"]
            ),
            "changelog_files": self.find_existing_files(
                repo_dir,
                ["CHANGELOG.md", "CHANGELOG.rst", "HISTORY.md", "RELEASE.md", "RELEASES.md"]
            ),
            "contributing_files": self.find_existing_files(
                repo_dir,
                ["CONTRIBUTING.md", "CONTRIBUTING.rst", ".github/CONTRIBUTING.md"]
            ),
            "security_files": self.find_existing_files(
                repo_dir,
                ["SECURITY.md", "security.md", ".github/SECURITY.md"]
            )
        }

    def detect_environment_files(self, repo_dir):
        return self.find_existing_files(
            repo_dir,
            [
                "requirements.txt",
                "environment.yml",
                "environment.yaml",
                "pyproject.toml",
                "setup.py",
                "setup.cfg",
                "Pipfile",
                "package.json",
                "DESCRIPTION",
                "renv.lock"
            ]
        )

    def detect_lock_files(self, repo_dir):
        return self.find_existing_files(
            repo_dir,
            [
                "Pipfile.lock",
                "poetry.lock",
                "uv.lock",
                "requirements.lock",
                "package-lock.json",
                "yarn.lock",
                "pnpm-lock.yaml",
                "conda-lock.yml",
                "conda-lock.yaml",
                "renv.lock"
            ]
        )

    def detect_container_files(self, repo_dir):
        return self.find_existing_files(
            repo_dir,
            [
                "Dockerfile",
                "docker-compose.yml",
                "docker-compose.yaml",
                ".devcontainer/devcontainer.json"
            ]
        )

    def detect_ci_files(self, repo_dir):
        ci_files = self.find_existing_files(
            repo_dir,
            [
                ".gitlab-ci.yml",
                "azure-pipelines.yml",
                "Jenkinsfile",
                "tox.ini",
                "noxfile.py"
            ]
        )

        workflows = repo_dir / ".github" / "workflows"
        if workflows.exists():
            for path in workflows.glob("*"):
                if path.suffix.lower() in {".yml", ".yaml"}:
                    try:
                        ci_files.append(str(path.relative_to(repo_dir)))
                    except Exception:
                        ci_files.append(str(path))

        return ci_files

    def detect_checksum_files(self, repo_dir):
        checksum_names = {
            "checksums.txt",
            "checksum.txt",
            "sha256sums.txt",
            "sha256sum.txt",
            "SHA256SUMS",
            "SHA256SUMS.txt",
            "hashes.txt"
        }

        found = self.find_files_by_name(repo_dir, checksum_names)

        for path in self.find_files_by_extensions(repo_dir, [".sha256", ".sha512", ".md5"]):
            try:
                found.append(str(path.relative_to(repo_dir)))
            except Exception:
                found.append(str(path))

        return found

    def collect_dependency_text(self, repo_dir):
        relevant_files = []

        dependency_names = {
            "requirements.txt",
            "environment.yml",
            "environment.yaml",
            "pyproject.toml",
            "setup.py",
            "setup.cfg",
            "Pipfile",
            "package.json",
            "Dockerfile",
            "docker-compose.yml",
            "docker-compose.yaml"
        }

        for root, dirs, files in os.walk(repo_dir):
            dirs[:] = [
                d for d in dirs
                if d not in {
                    ".git",
                    "__pycache__",
                    ".pytest_cache",
                    ".mypy_cache",
                    "node_modules",
                    ".venv",
                    "venv",
                    "build",
                    "dist"
                }
            ]

            for file in files:
                if file in dependency_names or file.lower() in dependency_names:
                    relevant_files.append(Path(root) / file)

        combined = ""

        for path in relevant_files[:100]:
            try:
                relative = path.relative_to(repo_dir)
            except Exception:
                relative = path

            combined += f"\n\n--- FILE: {relative} ---\n\n"
            combined += self.read_text_file(path)

        return combined

    def detect_version_pinning(self, repo_dir):
        text = self.collect_dependency_text(repo_dir)

        pinned_patterns = [
            r"==\s*[0-9]",
            r">=\s*[0-9]",
            r"<=\s*[0-9]",
            r"~=\s*[0-9]",
            r"@[a-f0-9]{7,40}",
            r"sha256:",
            r"FROM\s+[a-zA-Z0-9_\-/]+:[a-zA-Z0-9_\.\-]+"
        ]

        untrusted_patterns = [
            r"http://",
            r"git\+http://",
            r"curl\s+.*\|\s*(bash|sh)",
            r"wget\s+.*\|\s*(bash|sh)",
            r"pip\s+install\s+.*--trusted-host",
            r"pip\s+install\s+.*--extra-index-url\s+http://"
        ]

        pinned_matches = []

        for pattern in pinned_patterns:
            matches = re.findall(pattern, text, flags=re.IGNORECASE)
            pinned_matches.extend(matches[:20])

        untrusted_matches = []

        for pattern in untrusted_patterns:
            matches = re.findall(pattern, text, flags=re.IGNORECASE)
            if matches:
                untrusted_matches.append(pattern)

        return {
            "pinned_matches_count": len(pinned_matches),
            "pinned_examples": pinned_matches[:20],
            "untrusted_sources_count": len(untrusted_matches),
            "untrusted_source_patterns": untrusted_matches
        }

    def detect_release_or_version_metadata(self, repo_dir):
        version_files = self.find_existing_files(
            repo_dir,
            [
                "VERSION",
                "version.txt",
                "__version__.py",
                "pyproject.toml",
                "setup.py",
                "package.json"
            ]
        )

        version_evidence = []

        for relative in version_files:
            path = repo_dir / relative
            text = self.read_text_file(path, max_chars=50000)

            if re.search(r"version\s*=", text, flags=re.IGNORECASE):
                version_evidence.append(relative)
            elif re.search(r'"version"\s*:', text, flags=re.IGNORECASE):
                version_evidence.append(relative)
            elif relative.lower() in {"version", "version.txt"}:
                version_evidence.append(relative)

        return version_files, version_evidence

    def evaluate_integrity_provenance(self, repo_dir, artifact_data):
        config = artifact_data.get("integrity_provenance", {})

        minimum_score = config.get("minimum_score", self.minimum_score)
        max_untrusted_sources = config.get("max_untrusted_sources", self.max_untrusted_sources)

        metadata_files = self.detect_metadata_files(repo_dir)
        environment_files = self.detect_environment_files(repo_dir)
        lock_files = self.detect_lock_files(repo_dir)
        container_files = self.detect_container_files(repo_dir)
        ci_files = self.detect_ci_files(repo_dir)
        checksum_files = self.detect_checksum_files(repo_dir)
        version_files, version_evidence = self.detect_release_or_version_metadata(repo_dir)
        version_pinning = self.detect_version_pinning(repo_dir)

        has_git = self.has_git_metadata(repo_dir)
        git_commit = self.get_git_commit_hash(repo_dir)
        git_remote = self.get_git_remote_url(repo_dir)

        score = 0
        evidence = []
        issues = []

        if has_git and git_commit:
            score += 1
            evidence.append(f"Git metadata present with commit hash: {git_commit[:12]}")
        else:
            issues.append("No Git metadata or commit hash available")

        if git_remote:
            score += 1
            evidence.append(f"Git remote provenance available: {git_remote}")
        else:
            issues.append("No Git remote provenance available")

        if metadata_files["readme_files"]:
            score += 1
            evidence.append(f"README present: {', '.join(metadata_files['readme_files'])}")
        else:
            issues.append("No README found")

        if metadata_files["license_files"]:
            score += 1
            evidence.append(f"License metadata present: {', '.join(metadata_files['license_files'])}")
        else:
            issues.append("No LICENSE/COPYING file found")

        if metadata_files["citation_files"]:
            score += 1
            evidence.append(f"Citation/provenance metadata present: {', '.join(metadata_files['citation_files'])}")
        else:
            issues.append("No CITATION.cff or CODEMETA.json found")

        if environment_files:
            score += 1
            evidence.append(f"Environment/dependency files present: {', '.join(environment_files[:10])}")
        else:
            issues.append("No environment/dependency files found")

        if lock_files:
            score += 1
            evidence.append(f"Lock files present: {', '.join(lock_files[:10])}")
        else:
            issues.append("No lock file found")

        if container_files:
            score += 1
            evidence.append(f"Container/deployment files present: {', '.join(container_files[:10])}")
        else:
            issues.append("No container/deployment files found")

        if ci_files:
            score += 1
            evidence.append(f"CI/build workflow files present: {', '.join(ci_files[:10])}")
        else:
            issues.append("No CI/build workflow found")

        if checksum_files:
            score += 1
            evidence.append(f"Checksum/hash files present: {', '.join(checksum_files[:10])}")
        else:
            issues.append("No checksum/hash files found")

        if version_evidence:
            score += 1
            evidence.append(f"Version/release metadata present: {', '.join(version_evidence[:10])}")
        else:
            issues.append("No explicit version/release metadata found")

        if version_pinning["pinned_matches_count"] > 0:
            score += 1
            evidence.append(f"Version pinning or bounded versions detected: {version_pinning['pinned_matches_count']} matches")
        else:
            issues.append("No version pinning or bounded version evidence found")

        if version_pinning["untrusted_sources_count"] <= max_untrusted_sources:
            score += 1
            evidence.append(f"Untrusted dependency-source patterns within threshold: {version_pinning['untrusted_sources_count']} <= {max_untrusted_sources}")
        else:
            issues.append(
                f"Untrusted dependency-source patterns exceed threshold: "
                f"{version_pinning['untrusted_sources_count']} > {max_untrusted_sources}"
            )

        integrity_provenance = (
            score >= minimum_score
            and version_pinning["untrusted_sources_count"] <= max_untrusted_sources
        )

        return {
            "integrity_provenance": integrity_provenance,
            "score": score,
            "minimum_score": minimum_score,
            "has_git": has_git,
            "git_commit": git_commit,
            "git_remote": git_remote,
            "readme_files": metadata_files["readme_files"],
            "license_files": metadata_files["license_files"],
            "citation_files": metadata_files["citation_files"],
            "changelog_files": metadata_files["changelog_files"],
            "contributing_files": metadata_files["contributing_files"],
            "security_files": metadata_files["security_files"],
            "environment_files": environment_files,
            "lock_files": lock_files,
            "container_files": container_files,
            "ci_files": ci_files,
            "checksum_files": checksum_files,
            "version_files": version_files,
            "version_evidence": version_evidence,
            "pinned_matches_count": version_pinning["pinned_matches_count"],
            "pinned_examples": version_pinning["pinned_examples"],
            "untrusted_sources_count": version_pinning["untrusted_sources_count"],
            "untrusted_source_patterns": version_pinning["untrusted_source_patterns"],
            "evidence": evidence,
            "issues": issues
        }

    def check_artifact(self, artifact_id, artifact_data):
        title = artifact_data.get("title", "")
        uri = artifact_data.get("uri", "")

        print("\n" + "=" * 80)
        print(f"🔍 Integrity and Provenance Check for {artifact_id}")
        print(f"📦 Title: {title}")
        print(f"🔗 URI: {uri}")

        artifact_result = {
            "artifact_id": artifact_id,
            "title": title,
            "uri": uri,
            "integrity_provenance": False,
            "status": "failed"
        }

        if not self.is_git_repository(uri):
            print("❌ Unsupported artifact type for this checker.")
            artifact_result["reason"] = "Unsupported artifact type."
            return artifact_result

        repo_dir = self.clone_repository(artifact_id, uri)

        if repo_dir is None:
            artifact_result["reason"] = "Repository could not be cloned."
            artifact_result["status"] = "not_evaluated_repository_unavailable"
            return artifact_result

        result = self.evaluate_integrity_provenance(repo_dir, artifact_data)
        artifact_result.update(result)

        print("\n📊 Integrity and provenance evidence:")
        print(f" - Score: {result['score']} / required {result['minimum_score']}")
        print(f" - Git metadata: {'✅' if result['has_git'] else '❌'}")
        print(f" - Git commit: {result['git_commit'][:12] if result['git_commit'] else 'None'}")
        print(f" - README files: {', '.join(result['readme_files']) if result['readme_files'] else 'None'}")
        print(f" - License files: {', '.join(result['license_files']) if result['license_files'] else 'None'}")
        print(f" - Citation files: {', '.join(result['citation_files']) if result['citation_files'] else 'None'}")
        print(f" - Environment files: {', '.join(result['environment_files']) if result['environment_files'] else 'None'}")
        print(f" - Lock files: {', '.join(result['lock_files']) if result['lock_files'] else 'None'}")
        print(f" - Container files: {', '.join(result['container_files']) if result['container_files'] else 'None'}")
        print(f" - CI files: {', '.join(result['ci_files']) if result['ci_files'] else 'None'}")
        print(f" - Checksum files: {', '.join(result['checksum_files']) if result['checksum_files'] else 'None'}")
        print(f" - Version-pinning evidence count: {result['pinned_matches_count']}")
        print(f" - Untrusted source patterns: {result['untrusted_sources_count']}")

        print("\n🔎 Evidence found:")
        if result["evidence"]:
            for item in result["evidence"]:
                print(f" - {item}")
        else:
            print(" - No integrity/provenance evidence found.")

        print("\n⚠️ Issues / weak evidence:")
        if result["issues"]:
            for item in result["issues"]:
                print(f" - {item}")
        else:
            print(" - No major integrity/provenance issues detected.")

        if result["integrity_provenance"]:
            artifact_result["status"] = "passed"
            print("\n✅ Integrity and Provenance Result: PASSED")
        else:
            artifact_result["status"] = "failed"
            print("\n❌ Integrity and Provenance Result: FAILED")

        return artifact_result

    def run(self):
        self.results = []

        print("🌱 Starting Integrity and Provenance Fitness Function")
        print(f"📄 Metadata file: {self.json_file}")
        print(f"📁 Download directory: {self.download_dir}")

        for artifact_id, artifact_data in self.artifacts.items():
            result = self.check_artifact(artifact_id, artifact_data)
            self.results.append(result)

        print("\n" + "=" * 80)
        print("📌 Integrity and Provenance Summary")
        print("=" * 80)

        for result in self.results:
            icon = "✅" if result["integrity_provenance"] else "❌"
            print(f"{icon} {result['artifact_id']}: {result['status']}")

        return self.results

In [3]:
checker = IntegrityProvenanceChecker(
    json_file="artifacts.json",
    download_dir="downloads",
    minimum_score=5,
    max_untrusted_sources=0
)

integrity_provenance_results = checker.run()

🌱 Starting Integrity and Provenance Fitness Function
📄 Metadata file: artifacts.json
📁 Download directory: downloads

🔍 Integrity and Provenance Check for artifact_1
📦 Title: We provide our resources in a dedicated repository
🔗 URI: https://github.com/hihey54/hicss58
📁 Repository already exists: downloads/hicss58

📊 Integrity and provenance evidence:
 - Score: 7 / required 5
 - Git metadata: ✅
 - Git commit: 02f4d287a096
 - README files: README.md
 - License files: LICENSE
 - Citation files: None
 - Environment files: requirements.txt
 - Lock files: None
 - Container files: None
 - CI files: None
 - Checksum files: None
 - Version-pinning evidence count: 3
 - Untrusted source patterns: 0

🔎 Evidence found:
 - Git metadata present with commit hash: 02f4d287a096
 - Git remote provenance available: https://github.com/hihey54/hicss58
 - README present: README.md
 - License metadata present: LICENSE
 - Environment/dependency files present: requirements.txt
 - Version pinning or bounded vers

In [4]:
if PANDAS_AVAILABLE:
    df = pd.DataFrame(integrity_provenance_results)

    columns_to_show = [
        "artifact_id",
        "title",
        "integrity_provenance",
        "status",
        "score",
        "minimum_score",
        "has_git",
        "git_commit",
        "readme_files",
        "license_files",
        "citation_files",
        "environment_files",
        "lock_files",
        "container_files",
        "ci_files",
        "checksum_files",
        "pinned_matches_count",
        "untrusted_sources_count"
    ]

    existing_columns = [col for col in columns_to_show if col in df.columns]
    display(df[existing_columns])
else:
    for result in integrity_provenance_results:
        print(result)

,artifact_id,title,integrity_provenance,status,score,minimum_score,has_git,git_commit,readme_files,license_files,citation_files,environment_files,lock_files,container_files,ci_files,checksum_files,pinned_matches_count,untrusted_sources_count
0,artifact_1,We provide our resources in a dedicated reposi...,True,passed,7.0,5.0,True,02f4d287a0964f4badd39fb3d6613502bd195b74,[README.md],[LICENSE],[],[requirements.txt],[],[],[],[],3.0,0.0
1,artifact_2,Trending Customer Dataset,False,not_evaluated_repository_unavailable,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,artifact_3,Python algorithms,True,passed,11.0,5.0,True,791deb40f93c65a020f2c75e41b55c97ad566cc0,[README.md],[LICENSE.md],[],[pyproject.toml],[uv.lock],[.devcontainer/devcontainer.json],"[.github/workflows/project_euler.yml, .github/...",[],20.0,0.0
3,artifact_4,Scikit-learn,True,passed,10.0,5.0,True,ab6f86c1086377c19754177e0f46eb26e6282be5,[README.rst],[COPYING],[CITATION.cff],[pyproject.toml],[],[.devcontainer/devcontainer.json],"[.github/workflows/publish_pypi.yml, .github/w...",[],21.0,0.0
4,artifact_5,Pandas,True,passed,10.0,5.0,True,dc7aa11de3574662d718338f0d76a28c6b27c40d,[README.md],[LICENSE],[CITATION.cff],"[environment.yml, pyproject.toml]",[],[],"[.github/workflows/unit-tests.yml, .github/wor...",[],20.0,0.0
5,artifact_6,NumPy,True,passed,10.0,5.0,True,78e8ddee409f725d2fe1b0b24047d02aba8939a9,[README.md],[LICENSE.txt],[],"[environment.yml, pyproject.toml]",[],[.devcontainer/devcontainer.json],"[.github/workflows/dependency-review.yml, .git...",[],15.0,0.0
6,artifact_7,Matplotlib,True,passed,11.0,5.0,True,df808aae3e1b884c35ebfbd64ef58e7420958ff4,[README.md],[LICENSE],[CITATION.cff],"[environment.yml, pyproject.toml]",[],[.devcontainer/devcontainer.json],"[azure-pipelines.yml, tox.ini, .github/workflo...",[],22.0,0.0
7,artifact_8,Scrapy,True,passed,9.0,5.0,True,2d007bc4508423bbd57ee9017da22c16bbd576e1,[README.rst],[LICENSE],[],[pyproject.toml],[],[],"[tox.ini, .github/workflows/auto-close-llm-pr....",[],41.0,0.0
8,artifact_9,Flask,True,passed,11.0,5.0,True,7374c85ddefc3f4b177a698ab9f0cbb6a5c0b392,[README.md],[LICENSE.txt],[],[pyproject.toml],[uv.lock],[.devcontainer/devcontainer.json],"[.github/workflows/tests.yaml, .github/workflo...",[],29.0,0.0
9,artifact_10,TensorFlow,False,failed,7.0,6.0,True,bf296b0953cab3cf5fcf10603ddcc23ead5132f7,[README.md],[LICENSE],[CITATION.cff],[],[],[],"[.github/workflows/cffconvert.yml, .github/wor...",[],35.0,1.0


In [5]:
output_file = "integrity_provenance_results.json"

with open(output_file, "w", encoding="utf-8") as file:
    json.dump(integrity_provenance_results, file, indent=4)

print(f"✅ Results saved to {output_file}")

✅ Results saved to integrity_provenance_results.json
